### Practice Notebook
The purpose of this notebook is to relearn pandas so that i do not have to rely on vibe coding  
The journey starts again on 15th april 2026  
The Main goal is to gain enough proficiency with pandas to the point that i do not have to 

There are also a public sources for the documentation of pandas. This is for a quick reference  
Pandas API Reference  
This is the ultimate "dictionary." If you want to know every single argument for .groupby() or .merge(), this is the place. 
https://pandas.pydata.org/docs/reference/index.html

Pandas User Guide
Better for learning concepts. It explains the "why" behind things like Method Chaining or Time Series analysis.
https://pandas.pydata.org/docs/user_guide/index.html

Official Getting Started Tutorial
Short, punchy guides like "How do I read and write tabular data?"
https://pandas.pydata.org/docs/getting_started/index.html

Official Pandas Cheat Sheet (PDF)
https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf


In [2]:
# Data handling and graphics
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import scipy.stats as stats

# Analysis and Transformers
from scipy.stats import ttest_ind, ks_2samp, chi2_contingency, levene, pearsonr
from sklearn.feature_selection import mutual_info_classif ,mutual_info_regression

# Notebook Shortcuts

---

### **Execution & Saving**
| Shortcut | Action |
| :--- | :--- |
| **Shift + Enter** | Run the current cell and select the one below |
| **Ctrl + Enter** | Run the selected cells |
| **Alt + Enter** | Run the current cell and insert a new one below |
| **Ctrl + S** | Save and checkpoint |

---

### **Command Mode**
*Press `Esc` to enable (Cell border is blue)*

* **A**: Insert a new cell **above** the current cell.  
* **B**: Insert a new cell **below** the current cell.  
* **D, D**: **Delete** the selected cell.  
* **Z**: **Undo** cell deletion.  
* **C**: **Copy** selected cells.  
* **V**: **Paste** cells below.  
* **M**: Change the cell type to **Markdown**.  
* **Y**: Change the cell type to **Code**.  
* **1 to 6**: Change the cell to a **Heading** (Level 1-6).  
* **L**: Toggle **line numbers** on/off.  
* **Shift + M**: **Merge** selected cells or the cell below.  
* **Up / K**: Select the cell **above**.  
* **Down / J**: Select the cell **below**.  

---

### **Edit Mode**
*Press `Enter` to enable (Cell border is green)*

* **Tab**: Code completion or indent.  
* **Shift + Tab**: Show **Docstring** (documentation) for the object you just typed.  
* **Ctrl + /**: **Comment** or uncomment lines.  
* **Ctrl + Shift + -**: **Split** the current cell into two at the cursor.  
* **Ctrl + Z**: Undo.  
* **Ctrl + Home / End**: Go to the start or end of the cell.

#### Data Structure and filtering

While we normally encounter object dtypes often. its good to know that it often slows down the system due to it being inherently O(n)  
When a label has mixed dtypes, pandas will default the dtype to object meaning we should have as few object dtypes as possible.  
Using the `.is_unique` Property will see if the O(1) lookups are compromised if its false.  

So we should try and convert Object dtypes into something else
Repeated Strings -> categorical  
Binary strings (yes/no) -> bool or boolean  
Numerical strings ('1.5') -> float32 or int32  
Date strings - datetime64[ns]

Reduces memory and improves filtering logic
We can further reduce memory if the numbers are small enough (int64 to int8)  
This is typically just for the EDA phase.

For the final input into the models it might be better to upcast the variables into the native language.  
For example, when standard scaler processes a label, it is converted into float anyway.    
**We try to standardise dtype into int32 or float32 because its faster than the 64bit versions. Most GPUs are optimised for FP32, using float64 will double or quadruple the time without any meaningfull boost in accuracy.**
The final variables should lso be casted into their final data types.  
The Major reason why we upcast before feeding is because of ram, if the data has a lot of int8. If we didnt upcast it prior the model will process it to float32 the ram will incease a few fold and exceed the amount of ram available for training.  
We upcast so we know exactly how much memory is being used.  
**This also means that during modeling, we have to set the default dtype to float32 instead of float64**

The exception is forest based models as a smaller downcasted dtype, specifically int8 for lightgbm and xgboost since it histogram bins them into 256 bins by default.  So the model does not have to spend time processing the internal binning

For XGboost/light BGM they can handle categoricals, so there is no need to one hot encode (its often superior if fed categoricals)

#### Multi-Indexing (Hierarchical Data)
Hierarchical data is can add a lot to the visualisation of data.  
However whenever we get data its usually always viewed in a tabular form, we also eventually need to flatten the data for feeding into a machine learning model. 

So if only one table was loaded and theres a hierarchy, its often better to index the hierachy. Literally index the labels that constitute them so when we query for them its faster.

If the database already has 2 tables or more representing the hierarchy, we use merge to combine them before processing



#### Logical Indexing / boolean Indexing
At the base level its just passing a statement like x>30 and returning the rows that are true.

This is incredibly uninquitive.  
The only reason why its called indexing is because when you put something inside square brackets df[...], you are "indexing" into that object.  
When you put a Label in the brackets, you are doing a Lookup.  
When you put a Boolean Series (the mask) in the brackets, you are doing a Selection.  

If you're selecting specific Rows based on a property, you're using Logical Indexing.  
If you're selecting specific Columns by name, you're using Label Indexing.  

The reason why we do this is to make use of the inate Numpy code otherwise we have to run for loops (which is terrible)

loc vs iloc

column operations

Grouping and aggregation

group by object

pivot table

Summary stats mastering

Data cleaning and transformation

Handling Missing Values

Vectorizing String Components

Applying Features

#### Regular Expressions
Mainly used for matching and then transforming the matched  
I went down the rabbit hole with this for a while...

Regular Expressions are normally handed by python's re library  
Normally regrex=True  
When regex=False, it tells the pandas to bypass the re library and use pythons basic string methods  
Which can be faster as it ignores metacharacters (like '.')

There are also 2 other libraries called re2 and regex  
re2 is managed by Google, Mainly for security. Standard regex engines uses backtracking that can crash a server via ReDos so it uses different algorithms to manage some functions  
while regex is by a public developer Matthew Barnett to replace re and with better functionality for recursion and unicode

Normally although regex=True, its not using Regex at all but re.

More info can be read at https://www.regular-expressions.info/tools.html

In [ ]:
# Usual prepared code for cleaning
for col in df_data.select_dtypes(include=['object', 'string']):
        df_data[col] = (
            df_data[col]
            .str.lower()                            # Converts text to lowercase
            .str.strip()                            # Strip leading/trailing whitespace.
            .str.replace(r'\s+', ' ')               # Replace whitespaces that are one or more with 1 whitespace
            .str.replace(r'\s+', '', regex=True)    # Remove all whitespace (including between words)
            .str.replace('_', '', regex=False)      # Remove all underscores
            .str.replace('.', '', regex=False)      # Remove all periods
        )